# Capítulo 14 — Robótica: o rover autônomo

## Missão
Juntar tudo num **robô autônomo**. Você vai escrever uma interface de alto nível (andar, girar, medir a distância à frente) sobre o mapa, coordenar sensores e motores para não bater, dar ao rover um **cérebro** que planeja rotas (busca em largura) para coletar todas as amostras e voltar à base, e terminar com IA: um modelo treinado que olha a "câmera" e decide para onde virar para perseguir uma bola.

**No fim deste caderno você terá:** os módulos `meu_rover/robo.py` (interface do robô) e `meu_rover/cerebro.py` (planejador de missão), e um rover que completa a missão sozinho.

**Leitura essencial (opcional, livro em inglês):**
- Understanding the Main Parts of a Robot · PDF p. 594
- Building a Simple, High-Level Python Interface · PDF p. 619
- Coordinating Motor Movements with Sensors · PDF p. 635
- Making a Python Brain for Our Robot · PDF p. 639
- Machine Learning Using TensorFlow · PDF p. 651
- The FindAndChaseTheBall.py Python Program · PDF p. 673

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap14_robotica_o_rover_autonomo.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_rover/
os.makedirs("../meu_rover", exist_ok=True)
open("../meu_rover/__init__.py", "a").close()        # meu_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_rover")

## Ideia em 30 segundos
Todo robô tem:
- **sensores** (percebe);
- **controlador** (decide);
- **atuadores** (age);
- energia.

Uma boa interface de alto nível esconde os detalhes ("ande 3 passos" em vez de "PWM 0,6 por 2,3 s"). O cérebro fica em cima dela e roda o ciclo perceber → decidir → agir.

### 14.1 **Interface do robô.** Escreva `robo.py` com a classe `Robo(mapa, inicio=None)`:
- estado: `pos` (começa na base `B`), `direcao` (`"N"`, `"L"`, `"S"`, `"O"`), `bateria` (100), `coletadas` (conjunto) e `registro` (lista de ações);
- `girar_direita()` e `girar_esquerda()` (custam 0,2% de bateria);
- `distancia_frente()`: quantas células livres até uma rocha, olhando na direção atual (o "ultrassom");
- `frente(passos=1)`: anda até `passos` células, para antes de uma rocha, custa 1% por célula e devolve quantas andou;
- `coletar()`: se estiver sobre um `A` ainda não coletado, guarda a posição;
- `desenhar()`: o mapa com a seta do robô (`^ > v <`) e as amostras já coletadas como `x`.

_A célula grava `meu_rover/robo.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_rover/robo.py


### 14.2 **Teste a interface.** Crie o robô, gire para o Sul, meça a distância à frente, ande 10 (ele deve parar antes da rocha), gire para o Leste, ande 2, e mostre o desenho, a bateria e o registro.

### 14.3 **Sensores + motores: seguir a parede.** Implemente a regra da mão direita: a cada ciclo, se a direita estiver livre, vira à direita e anda; senão, se a frente estiver livre, anda; senão, vira à esquerda. Rode 60 ciclos coletando as amostras pelo caminho. Quantas o rover pegou e quanta bateria sobrou?

### 14.4 **O cérebro.** Escreva `cerebro.py` com:
- `caminho(mapa, inicio, fim)`: busca em largura (BFS, com `collections.deque`) e devolve a lista de posições do menor caminho (ou `None`);
- `para_comandos(caminho)`: transforma posições em direções (`"N"`, `"S"`, `"L"`, `"O"`);
- `executar(robo, direcoes)`: gira o robô até a direção certa e anda 1 a cada passo;
- `missao_completa(robo)`: visita as amostras sempre indo para a mais próxima **pelo caminho real**, coleta cada uma, volta à base e devolve um resumo (coletadas, passos, bateria).

_A célula grava `meu_rover/cerebro.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_rover/cerebro.py


### 14.5 **Missão autônoma.** Rode `missao_completa` num robô novo e compare com o "seguir a parede": quantas amostras, quantos passos e quanta bateria sobrou. Mostre o desenho final.

### 14.6 **IA no robô: a câmera.** Simule a câmera do rover: uma imagem de 1×30 pixels (brilho de 0 a 1) com ruído e, às vezes, uma "bola" clara (uma faixa de 4 pixels com brilho ~0,9) em uma posição aleatória. O rótulo é `esquerda`, `frente` ou `direita` (terço da imagem onde está o centro da bola) ou `nenhuma`. Gere 3.000 exemplos com semente fixa.

### 14.7 **Treine o "olho" do robô.** Treine um `RandomForestClassifier(n_estimators=100, random_state=0)` com 80% dos dados e mostre a acurácia no restante.

### 14.8 **Persiga a bola.** Simule a perseguição: a bola começa num ângulo aleatório entre -90° e 90° em relação à frente do robô. A câmera vê o ângulo como posição na imagem (`c = 15 + ângulo / 6`; fora de 0–29 = não vê). A cada ciclo o modelo decide: `esquerda` (gira +15°), `direita` (-15°), `frente` (avança: a distância cai 20%) ou `nenhuma` (gira 30° procurando). Considere que pegou quando a distância < 0,3. Rode 5 perseguições e mostre em quantos ciclos cada uma terminou.

## E com um robô de verdade?
A arquitetura é a mesma: `robo.py` passa a chamar a biblioteca do fabricante (no livro, o kit PiCar-B), em vez de mexer na grade, e o "olho" passa a receber os quadros de uma câmera real reduzidos para poucos pixels. Como o seu cérebro (`cerebro.py`) só usa a interface de alto nível, ele **não precisa mudar**. Esse é o maior ganho de separar as camadas.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
